# ENGR 521 Housing Price Prediction Project
## Optimization of Regression Models for King County House Sale Prices

**Team:** Anh Vu & Matthew Kouthong

**Dataset:** [King County House Sales](https://www.kaggle.com/datasets/harlfoxem/housesalesprediction)

**Models:** Linear Regression, Random Forest, XGBoost


### Introduction
Housing price prediction is a supervised machine learning regression problem that aims to estimate house sale prices based on features such as size, number of bedrooms and bathrooms, year built, renovation status, and location, where regression models are used because the output (price) is continuous. In this project, we use three models: Linear Regression, Random Forest Regression, and XGBoost (Extreme Gradient Boosting). We chose this topic because housing price prediction is a practical, real-world problem, and we are curious about how different machine learning models compare in performance. This project allows us to learn and apply regression methods on structured data through model training, evaluation, and comparison.

### Goal
The primary goal of this project is to build and evaluate regression models for housing price prediction using the King County dataset (and potential Snohomish County dataset) that are found on Kaggle. We will compare three models: Linear Regression (baseline model), Random Forest Regression (ensemble decision-tree model), and XGBoost (a boosting-based model that improves predictions by correcting previous errors). Each model will take housing attributes as input and output predicted house prices.
In addition to model comparison, we will focus on feature subset sampling and hyperparameter tuning. Instead of using all available features, we will test smaller optimized subsets of features to evaluate how feature sparsity affects performance. We will also tune key parameters for Random Forest and XGBoost to improve accuracy. Model performance will be evaluated using MAE, RMSE, and R2 score, where lower error and higher R2 indicate better performance. Success is defined as identifying the best-performing combination of model type, feature subset, and tuning strategy.

In [ ]:
# Setup
import pandas as pd

In [7]:
# Load Data
DATA_PATH = "kc_house_data.csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head(20))
display(df.columns)

Shape: (21613, 21)


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503
5,7237550310,20140512T000000,1225000.0,4,4.50,5420,101930,1.0,0,0,...,11,3890,1530,2001,0,98053,47.6561,-122.005,4760,101930
6,1321400060,20140627T000000,257500.0,3,2.25,1715,6819,2.0,0,0,...,7,1715,0,1995,0,98003,47.3097,-122.327,2238,6819
7,2008000270,20150115T000000,291850.0,3,1.50,1060,9711,1.0,0,0,...,7,1060,0,1963,0,98198,47.4095,-122.315,1650,9711
8,2414600126,20150415T000000,229500.0,3,1.00,1780,7470,1.0,0,0,...,7,1050,730,1960,0,98146,47.5123,-122.337,1780,8113
9,3793500160,20150312T000000,323000.0,3,2.50,1890,6560,2.0,0,0,...,7,1890,0,2003,0,98038,47.3684,-122.031,2390,7570


Index(['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living',
       'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade',
       'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode',
       'lat', 'long', 'sqft_living15', 'sqft_lot15'],
      dtype='object')

In [9]:
# Clean Data



# Outliers
df_clean = df.copy()
# grade: Houses normally have grade between 1 and 13 -> remove unrealistic values
df_clean = df_clean[df_clean['grade'] >= 1]
df_clean = df_clean[df_clean['grade'] <= 13]

# sqft_above: Very large houses (>6500 sqft) are rare and can skew the model -> remove them
df_clean = df_clean[df_clean['sqft_above'] > 0]
df_clean = df_clean[df_clean['sqft_above'] < 6500]

# sqft_basement: Basement should not be negative and extremely large basements are uncommon -> remove them
df_clean = df_clean[df_clean['sqft_basement'] >= 0]
df_clean = df_clean[df_clean['sqft_basement'] < 3000]

# yr_built: Year built must be positive -> remove invalid data
df_clean = df_clean[df_clean['yr_built'] > 0]

# yr_renovated: 0 means no renovation and remove negative value
df_clean = df_clean[df_clean['yr_renovated'] >= 0]

# zipcode: Zipcode is a category (location), not a numeric value -> convert to string
df_clean['zipcode'] = df_clean['zipcode'].astype(str)

# lat: Latitude should be within King County area -> remove wrong location data
df_clean = df_clean[df_clean['lat'] > 47]
df_clean = df_clean[df_clean['lat'] < 48]

# long: Longitude should be within King County area -> remove wrong location data
df_clean = df_clean[df_clean['long'] > -123]
df_clean = df_clean[df_clean['long'] < -121]

# sqft_living15: Nearby living area should be positive -> remove invalid values
df_clean = df_clean[df_clean['sqft_living15'] > 0]

# sqft_lot15: Extremely large lot sizes (>500000 sqft) are rare and can distort analysis -> remove them
df_clean = df_clean[df_clean['sqft_lot15'] > 0]
df_clean = df_clean[df_clean['sqft_lot15'] < 500000]

print(df_clean.shape)
df_clean.head(20)

(21594, 21)


,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,20141013T000000,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,20141209T000000,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,20150225T000000,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,20141209T000000,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,20150218T000000,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503
5,7237550310,20140512T000000,1225000.0,4,4.50,5420,101930,1.0,0,0,...,11,3890,1530,2001,0,98053,47.6561,-122.005,4760,101930
6,1321400060,20140627T000000,257500.0,3,2.25,1715,6819,2.0,0,0,...,7,1715,0,1995,0,98003,47.3097,-122.327,2238,6819
7,2008000270,20150115T000000,291850.0,3,1.50,1060,9711,1.0,0,0,...,7,1060,0,1963,0,98198,47.4095,-122.315,1650,9711
8,2414600126,20150415T000000,229500.0,3,1.00,1780,7470,1.0,0,0,...,7,1050,730,1960,0,98146,47.5123,-122.337,1780,8113
9,3793500160,20150312T000000,323000.0,3,2.50,1890,6560,2.0,0,0,...,7,1890,0,2003,0,98038,47.3684,-122.031,2390,7570


In [ ]:
# Helper Functions

In [ ]:
# Anh Feature Subset Experiments

In [ ]:
# Matthew Feature Subset Experiments

In [ ]:
# Base Line: Linear Regression

In [ ]:
# Random Forest

In [ ]:
# XGBoost

In [ ]:
# Combine Results and Visualization (Plot Maybe)

### Conclusions

#### Best model


#### Feature subset effect


#### Hyperparameter tuning effect


#### Linear vs nonlinear models


#### Limitations


#### Future development
